# Phase 3: Model Training and Evaluation

## Objective
Build and evaluate machine learning models (Random Forest and XGBoost) for job-resume matching prediction.

### Components
1. Load engineered features from Phase 2
2. Train Random Forest model with hyperparameter tuning
3. Train XGBoost model with hyperparameter tuning
4. Perform 5-fold cross-validation
5. Comprehensive evaluation: R², RMSE, Precision@K, NDCG
6. Feature importance analysis
7. Model comparison and selection

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

from pathlib import Path
import sys
sys.path.insert(0, '/workspace')

from src.data_loader import DataLoader
from src.data_cleaner import DataCleaner
from src.embeddings import generate_all_embeddings
from src.feature_engineering import engineer_all_features
from src.model_training import ModelTrainer, train_all_models
from src.evaluation_metrics import EvaluationMetrics, MetricsReporter

print("✓ All imports successful")

## 2. Data Loading and Preparation

In [ ]:
# Load and clean data
print("Loading data...")
loader = DataLoader()
jobs_raw, resumes_raw = loader.load_data()
print(f"Raw data: {len(jobs_raw)} jobs, {len(resumes_raw)} resumes")

# Clean data
print("\nCleaning data...")
cleaner = DataCleaner()
jobs_clean = cleaner.clean_jobs(jobs_raw)
resumes_clean = cleaner.clean_resumes(resumes_raw)
print(f"Clean data: {len(jobs_clean)} jobs, {len(resumes_clean)} resumes")

print("\n✓ Data preparation complete")

## 3. Feature Engineering

In [ ]:
# Check for cached features
features_path = Path("data/processed/features_engineered.csv")

if features_path.exists():
    print("Loading cached features...")
    features_df = pd.read_csv(features_path)
    print(f"✓ Features loaded: {features_df.shape}")
else:
    print("Generating features...")
    features_df = engineer_all_features(jobs_clean, resumes_clean)
    features_df.to_csv(features_path, index=False)
    print(f"✓ Features generated and cached: {features_df.shape}")

# Display feature info
print(f"\nFeature columns: {list(features_df.columns)}")
print(f"Match score range: [{features_df['match_score'].min():.4f}, {features_df['match_score'].max():.4f}]")
print(f"\nFeature statistics:")
features_df.describe()

## 4. Model Training with Hyperparameter Tuning

In [ ]:
print("Training models with hyperparameter tuning...")
print("This may take several minutes...\n")

training_results = train_all_models(features_df, hyperparameter_tuning=True)
trainer = training_results['trainer']
best_model_name = training_results['best_model_name']

print(f"\n✓ Models trained")
print(f"Best model: {best_model_name}")

## 5. Comprehensive Metrics Comparison

In [ ]:
# Create comparison dataframe
metrics_comparison = pd.DataFrame(trainer.metrics).T

print("\nMODEL COMPARISON - ALL METRICS")
print("=" * 80)
print(metrics_comparison.round(4).to_string())

# Highlight best model
print(f"\n✓ Best Model: {best_model_name.upper()}")
print(f"  R² Score: {metrics_comparison.loc[best_model_name, 'r2']:.4f}")
print(f"  RMSE: {metrics_comparison.loc[best_model_name, 'rmse']:.4f}")
print(f"  MAE: {metrics_comparison.loc[best_model_name, 'mae']:.4f}")

## 6. Visualization: Metrics Comparison

In [ ]:
# Plot key metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Model Comparison - Key Metrics', fontsize=16, fontweight='bold')

metrics_to_plot = ['r2', 'rmse', 'mae', 'precision@5', 'recall@5', 'ndcg@5']
colors = ['#FF6B6B', '#4ECDC4']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    models = list(trainer.metrics.keys())
    values = [trainer.metrics[model].get(metric, 0) for model in models]
    
    bars = ax.bar(models, values, color=colors[:len(models)], alpha=0.7, edgecolor='black')
    ax.set_title(metric.upper(), fontweight='bold')
    ax.set_ylabel('Score')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('data/visualizations/model_comparison_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison visualization saved")

## 7. Cross-Validation Analysis

In [ ]:
# Display cross-validation results
print("\nCROSS-VALIDATION RESULTS (5-Fold)")
print("=" * 60)

for model_name, cv_scores in trainer.cv_scores.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Fold scores: {cv_scores}")
    print(f"  Mean: {cv_scores.mean():.4f}")
    print(f"  Std Dev: {cv_scores.std():.4f}")
    print(f"  Min: {cv_scores.min():.4f}")
    print(f"  Max: {cv_scores.max():.4f}")

## 8. CV Scores Visualization

In [ ]:
# Plot CV scores
fig, ax = plt.subplots(figsize=(10, 6))

models = list(trainer.cv_scores.keys())
cv_data = [trainer.cv_scores[model] for model in models]

bp = ax.boxplot(cv_data, labels=models, patch_artist=True)

# Style the plot
colors = ['#FF6B6B', '#4ECDC4']
for patch, color in zip(bp['boxes'], colors[:len(models)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Cross-Validation Scores Distribution (5-Fold)', fontsize=14, fontweight='bold')
ax.set_ylabel('R² Score', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/visualizations/cross_validation_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Cross-validation visualization saved")

## 9. Feature Importance Analysis

In [ ]:
# Get feature importances for best model
best_model = trainer.models[best_model_name]
feature_cols = training_results['feature_cols']

importances = best_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print(f"\n{best_model_name.upper()} - FEATURE IMPORTANCE")
print("=" * 60)
print(feature_importance_df.to_string(index=False))

# Calculate cumulative importance
feature_importance_df['cumulative_importance'] = feature_importance_df['importance'].cumsum()
print(f"\nCumulative importance (top 5): {feature_importance_df.head()['cumulative_importance'].values[-1]:.4f}")

## 10. Feature Importance Visualization

In [ ]:
# Plot feature importance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot of top features
top_features = feature_importance_df.head(10)
ax1.barh(range(len(top_features)), top_features['importance'].values, color='#4ECDC4', alpha=0.7, edgecolor='black')
ax1.set_yticks(range(len(top_features)))
ax1.set_yticklabels(top_features['feature'].values)
ax1.set_xlabel('Importance', fontweight='bold')
ax1.set_title(f'{best_model_name.upper()} - Top 10 Features', fontweight='bold')
ax1.invert_yaxis()
ax1.grid(True, alpha=0.3, axis='x')

# Cumulative importance plot
cumsum_importance = feature_importance_df['cumulative_importance'].values
ax2.plot(range(len(cumsum_importance)), cumsum_importance, marker='o', linewidth=2, markersize=8, color='#FF6B6B')
ax2.axhline(y=0.95, color='green', linestyle='--', label='95% Threshold')
ax2.fill_between(range(len(cumsum_importance)), cumsum_importance, alpha=0.3, color='#FF6B6B')
ax2.set_xlabel('Feature Index', fontweight='bold')
ax2.set_ylabel('Cumulative Importance', fontweight='bold')
ax2.set_title('Cumulative Feature Importance', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/visualizations/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature importance visualization saved")

## 11. Prediction Analysis

In [ ]:
# Analyze predictions for best model
y_test = trainer.predictions[best_model_name]['y_test']
y_pred = trainer.predictions[best_model_name]['y_pred']

# Residuals
residuals = y_test - y_pred

print(f"\n{best_model_name.upper()} - PREDICTION ANALYSIS")
print("=" * 60)
print(f"\nPredictions Summary:")
print(f"  Range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print(f"  Mean: {y_pred.mean():.4f}")
print(f"  Std Dev: {y_pred.std():.4f}")

print(f"\nActual (Test Set) Summary:")
print(f"  Range: [{y_test.min():.4f}, {y_test.max():.4f}]")
print(f"  Mean: {y_test.mean():.4f}")
print(f"  Std Dev: {y_test.std():.4f}")

print(f"\nResiduals Summary:")
print(f"  Mean: {residuals.mean():.6f}")
print(f"  Std Dev: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## 12. Residual Analysis Visualization

In [ ]:
# Residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{best_model_name.upper()} - Residual Analysis', fontsize=14, fontweight='bold')

# 1. Actual vs Predicted
ax = axes[0, 0]
ax.scatter(y_test, y_pred, alpha=0.5, s=30, color='#4ECDC4')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Values', fontweight='bold')
ax.set_ylabel('Predicted Values', fontweight='bold')
ax.set_title('Actual vs Predicted', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Residuals vs Predicted
ax = axes[0, 1]
ax.scatter(y_pred, residuals, alpha=0.5, s=30, color='#FF6B6B')
ax.axhline(y=0, color='black', linestyle='--', lw=2)
ax.set_xlabel('Predicted Values', fontweight='bold')
ax.set_ylabel('Residuals', fontweight='bold')
ax.set_title('Residuals vs Predicted', fontweight='bold')
ax.grid(True, alpha=0.3)

# 3. Residuals histogram
ax = axes[1, 0]
ax.hist(residuals, bins=30, color='#FFD93D', alpha=0.7, edgecolor='black')
ax.axvline(x=residuals.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {residuals.mean():.4f}')
ax.set_xlabel('Residuals', fontweight='bold')
ax.set_ylabel('Frequency', fontweight='bold')
ax.set_title('Residuals Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 4. Q-Q plot approximation
ax = axes[1, 1]
sorted_residuals = np.sort(residuals)
theoretical_quantiles = np.sort(np.random.normal(0, residuals.std(), len(residuals)))
ax.scatter(theoretical_quantiles, sorted_residuals, alpha=0.5, s=30, color='#6BCB77')
ax.plot([sorted_residuals.min(), sorted_residuals.max()], 
        [sorted_residuals.min(), sorted_residuals.max()], 'r--', lw=2)
ax.set_xlabel('Theoretical Quantiles', fontweight='bold')
ax.set_ylabel('Sample Quantiles', fontweight='bold')
ax.set_title('Q-Q Plot (Normal Distribution)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/visualizations/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Residual analysis visualization saved")

## 13. Summary Report

In [ ]:
print("\n" + "=" * 80)
print("PHASE 3: MODEL TRAINING - FINAL SUMMARY")
print("=" * 80)

print(f"\n📊 DATASET STATISTICS")
print(f"  • Total Feature Pairs: {len(features_df):,}")
print(f"  • Features per Pair: {features_df.shape[1]}")
print(f"  • Match Score Range: [{features_df['match_score'].min():.4f}, {features_df['match_score'].max():.4f}]")

print(f"\n🎯 BEST MODEL: {best_model_name.upper()}")
print(f"  • R² Score: {trainer.metrics[best_model_name]['r2']:.4f}")
print(f"  • RMSE: {trainer.metrics[best_model_name]['rmse']:.4f}")
print(f"  • MAE: {trainer.metrics[best_model_name]['mae']:.4f}")
print(f"  • Precision@5: {trainer.metrics[best_model_name].get('precision@5', 'N/A')}")
print(f"  • NDCG@5: {trainer.metrics[best_model_name].get('ndcg@5', 'N/A')}")

print(f"\n✨ TOP 3 IMPORTANT FEATURES")
for idx, row in feature_importance_df.head(3).iterrows():
    print(f"  {idx+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n📈 CROSS-VALIDATION (5-Fold)")
cv_scores = trainer.cv_scores[best_model_name]
print(f"  • Mean R² Score: {cv_scores.mean():.4f}")
print(f"  • Std Dev: {cv_scores.std():.4f}")

print(f"\n💾 MODEL SAVED")
print(f"  • Location: models/best_{best_model_name}_model.pkl")

print("\n" + "=" * 80)
print("✅ Phase 3 Complete! Ready for Phase 4: Model Deployment")
print("=" * 80 + "\n")